# JARVIS-DFT Half-Heusler Thermoelectric Data -> SISSO-Ready Dataset

**Target (y):** Seebeck coefficient (n- and p-type combined via long format)

**Input variables (X):**
- carrier_sign (-1 = n-type, +1 = p-type)
- mean electronegativity
- mean valence electron count
- mean covalent radius
- electronegativity range
- band gap (optb88vdw_bandgap)
- mean melting point
- mean unfilled orbitals
- ehull (stability, filter/sanity use)

Seebeck reference point in JARVIS: 600 K, 10^20 cm^-3 doping. n-type negative, p-type positive.
Half-Heusler = ternary (3-element), 1:1:1 stoichiometry, space group F-43m (#216).

Run top to bottom in Google Colab.

## 1. Install dependencies

In [ ]:
!pip install jarvis-tools pandas matminer pymatgen -q

## 2. Download JARVIS dft_3d

In [ ]:
import pandas as pd
from jarvis.db.figshare import data as jdata

d = jdata("dft_3d")
df = pd.DataFrame(d)
print(f"Total entries: {len(df)}")

## 3. Elements + TE-data flag

In [ ]:
def get_elements(row):
    try:
        return set(row["atoms"]["elements"])
    except Exception:
        return set()

df["elements"] = df.apply(get_elements, axis=1)
df["n_elements"] = df["elements"].apply(len)

def has_te_data(row):
    return (row.get("n-Seebeck", "na") not in ("na", None)) or (row.get("p-Seebeck", "na") not in ("na", None))

df["has_te_data"] = df.apply(has_te_data, axis=1)

keep_cols = ["jid", "formula", "spg_number", "n-Seebeck", "p-Seebeck", "optb88vdw_bandgap", "ehull"]
keep_cols = [c for c in keep_cols if c in df.columns]

## 4. Filter for Half-Heusler family only

In [ ]:
def is_half_heusler(row):
    if row["n_elements"] != 3:
        return False
    try:
        if int(row.get("spg_number", -1)) != 216:
            return False
    except (ValueError, TypeError):
        return False
    comp = row["atoms"]["elements"]
    counts = {e: comp.count(e) for e in set(comp)}
    vals = list(counts.values())
    return len(vals) == 3 and max(vals) == min(vals)

hh_mask = df.apply(is_half_heusler, axis=1) & df["has_te_data"]
df_hh = df.loc[hh_mask, keep_cols].reset_index(drop=True)
print(f"Half-Heusler entries with TE data: {len(df_hh)}")

## 5. Reshape to long format (n/p -> carrier_sign)

In [ ]:
df_long = pd.melt(
    df_hh,
    id_vars=[c for c in df_hh.columns if c not in ["n-Seebeck", "p-Seebeck"]],
    value_vars=["n-Seebeck", "p-Seebeck"],
    var_name="doping_type",
    value_name="Seebeck"
)
df_long["doping_type"] = df_long["doping_type"].str.replace("-Seebeck", "")
df_long = df_long[df_long["Seebeck"] != "na"].copy()
df_long["Seebeck"] = df_long["Seebeck"].astype(float)
df_long["carrier_sign"] = df_long["doping_type"].map({"n": -1, "p": 1})

print(f"Long-format entries: {len(df_long)}")

## 6. Compute Magpie elemental descriptors, keep only the agreed variables

In [ ]:
from matminer.featurizers.composition import ElementProperty
from pymatgen.core import Composition

featurizer = ElementProperty.from_preset("magpie")
feature_labels = featurizer.feature_labels()

def safe_composition(formula):
    try:
        return Composition(formula)
    except Exception:
        return None

df_long["composition"] = df_long["formula"].apply(safe_composition)
df_long = df_long[df_long["composition"].notna()].reset_index(drop=True)

def featurize_row(comp):
    try:
        return featurizer.featurize(comp)
    except Exception:
        return [None] * len(feature_labels)

features = df_long["composition"].apply(featurize_row)
features_df = pd.DataFrame(features.tolist(), columns=feature_labels, index=df_long.index)
df_full = pd.concat([df_long.drop(columns=["composition"]), features_df], axis=1)
df_full = df_full.dropna(subset=feature_labels)

selected = [
    "MagpieData mean Electronegativity",
    "MagpieData mean NValence",
    "MagpieData mean CovalentRadius",
    "MagpieData range Electronegativity",
    "MagpieData mean MeltingT",
    "MagpieData mean NUnfilled",
]
selected = [c for c in selected if c in df_full.columns]

final_cols = ["jid", "formula", "doping_type", "carrier_sign",
              "optb88vdw_bandgap", "ehull", "Seebeck"] + selected
final_cols = [c for c in final_cols if c in df_full.columns]

df_sisso = df_full[final_cols].dropna().reset_index(drop=True)
print(f"Final SISSO-ready rows: {len(df_sisso)}")
print(f"Input variables: {[c for c in df_sisso.columns if c not in ['jid','formula','doping_type','Seebeck']]}")
df_sisso.head()

## 7. Save CSV

In [ ]:
df_sisso.to_csv("half_heusler_SISSO_ready.csv", index=False)
print("Saved: half_heusler_SISSO_ready.csv")

## 8. (Optional) Download

In [ ]:
from google.colab import files
files.download("half_heusler_SISSO_ready.csv")